<a href="https://colab.research.google.com/github/masonsears/-Data-Processing-Visualization-CPSMA-4313-01-projects/blob/Projects/ProjectRecreatingMasters_MasonSears.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
import json
import requests
import plotly.graph_objects as go
from bs4 import BeautifulSoup

In [10]:
url = 'https://www.sankeyart.com/sankeys/149306/'
response = requests.get(url)
html_content = response.text

soup = BeautifulSoup(html_content, 'html.parser')
element = soup.select_one('#initial-data')

data = json.loads(element.string)
sankey_data = data['sankey']['chart']['data']
print(sankey_data)

{'flows': [{'id': 'flow-0', 'color': '#696969', 'value': 209.586, 'source_node_id': 'node-867e57bd062c7169995dc03cc0541c19', 'target_node_id': 'node-068f80c7519d0528fb08e82137a72131', 'value_comparison': 201.183, 'data_grid_row_index': 0}, {'id': 'flow-1', 'color': '#696969', 'value': 33.708, 'source_node_id': 'node-1748c0644a50090814d3e170723ccc5c', 'target_node_id': 'node-068f80c7519d0528fb08e82137a72131', 'value_comparison': 29.984, 'data_grid_row_index': 1}, {'id': 'flow-2', 'color': '#696969', 'value': 28.023, 'source_node_id': 'node-1b9018182a49e16ba85bb095f224867c', 'target_node_id': 'node-068f80c7519d0528fb08e82137a72131', 'value_comparison': 26.694, 'data_grid_row_index': 2}, {'id': 'flow-3', 'color': '#696969', 'value': 35.686, 'source_node_id': 'node-1767478556da307a5a6132afe6ccad94', 'target_node_id': 'node-068f80c7519d0528fb08e82137a72131', 'value_comparison': 37.005, 'data_grid_row_index': 3}, {'id': 'flow-4', 'color': '#696969', 'value': 307.003, 'source_node_id': 'node-

In [11]:
flow_details = []
for flow in sankey_data['flows']:
    source_node_name = "N/A"
    target_node_name = "N/A"
    for node in sankey_data['nodes']:
        if node['id'] == flow['source_node_id']:
            source_node_name = node['name']
        if node['id'] == flow['target_node_id']:
            target_node_name = node['name']
    flow_details.append({
        'Source': source_node_name,
        'Target': target_node_name,
        'Value': flow['value']
    })

df_flows = pd.DataFrame(flow_details)
display(df_flows)

,Source,Target,Value
0,iPhone,Products,209.586
1,Mac,Products,33.708
2,iPad,Products,28.023
3,Wearable & acc.,Products,35.686
4,Products,Revenue,307.003
5,Services,Revenue,109.158
6,Revenue,Gross profit,195.201
7,Revenue,Cost of revenue,220.960
8,Gross profit,Operating profit,133.050
9,Gross profit,Operating expenses,62.151


In [12]:
sankey_data = data['sankey']['chart']['data']

# Color maping, changing the colors to be the apple color pallet rather than the ugly colors they chose
color_map = {
    '#00A34C': '#0088cc',  # green to blue
    '#D1003F': '#eeeeee'    # red to light gray
}

# applying th color change
for flow in sankey_data['flows']:
    original_color = flow['color'].strip()
    if original_color in color_map:
        flow['color'] = color_map[original_color]

node_labels = [node['name'] for node in sankey_data['nodes']]
node_id_to_index = {node['id']: i for i, node in enumerate(sankey_data['nodes'])}

source_nodes = [node_id_to_index[flow['source_node_id']] for flow in sankey_data['flows']]
target_nodes = [node_id_to_index[flow['target_node_id']] for flow in sankey_data['flows']]
flow_values = [flow['value'] for flow in sankey_data['flows']]
flow_colors = [flow['color'] for flow in sankey_data['flows']]

node_source_colors = {}
for flow in sankey_data['flows']:
    if flow['source_node_id'] not in node_source_colors:
        node_source_colors[flow['source_node_id']] = flow['color']

# Gray as the default color like the original
node_colors_plotly = [node_source_colors.get(node['id'], 'gray') for node in sankey_data['nodes']]

fig = go.Figure(data=[
    go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=node_labels,
            color=node_colors_plotly,
            hovertemplate='<b>Node:</b> %{label}<br><b>Value:</b> $%{value}B<extra></extra>' # display node name and value when hovering over the node
        ),
        link=dict(
            source=source_nodes,
            target=target_nodes,
            value=flow_values,
            color=flow_colors,
            hovertemplate='<b>From:</b> %{source.label}<br><b>To:</b> %{target.label}<br><b>Value:</b> $%{value}B<extra></extra>' # Displaying more stuff
        )
    )
])

fig.update_layout(title_text="Sankey Diagram", font_size=10)
fig.show()